# LEGO Brick Color Detector

Run every cell from top to bottom (menu bar -> Run -> Run All Cells).
An **upload button** will appear below the last cell — click it, choose a photo
of LEGO bricks from your computer, and the result (boxes drawn + color counts)
will appear right underneath.

No account or sign-in needed — this notebook runs entirely in your browser via Binder.


In [ ]:
%matplotlib inline
from collections import Counter
import matplotlib.pyplot as plt
from ultralytics import YOLO

MODEL_PATH = 'models/lego_model.pt'
model = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")
print(f"Classes: {model.names}")

def count_legos(image_path, conf=0.5):
    results = model(image_path, conf=conf, verbose=False)
    r = results[0]
    counts = Counter(r.names[int(c)] for c in r.boxes.cls.tolist())
    total = sum(counts.values())

    plt.figure(figsize=(10, 10))
    plt.imshow(r.plot()[..., ::-1])
    plt.axis('off')
    title = f"Total: {total}  |  " + "  ".join(f"{k}: {v}" for k, v in sorted(counts.items()))
    plt.title(title, fontsize=13)
    plt.show()

    print(f"\n=== LEGO Count ===")
    print(f"Total bricks: {total}")
    for color, n in sorted(counts.items()):
        print(f"  {color:8s}: {n}")
    return dict(counts)


In [ ]:
# Upload button (works in the browser, no account needed)
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

uploader = widgets.FileUpload(accept='image/*', multiple=False, description='Upload photo')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output(wait=True)
        if not uploader.value:
            return
        # ipywidgets 8: uploader.value is a tuple of dicts; ipywidgets 7: a dict keyed by filename
        item = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
        filename = item['name'] if isinstance(uploader.value, dict) else item.name
        content = item['content'] if isinstance(uploader.value, dict) else item.content

        with open(filename, 'wb') as f:
            f.write(bytes(content))

        print(f"Received: {filename}")
        count_legos(filename)

uploader.observe(on_upload_change, names='value')
display(uploader, output)
